In [12]:
import pandas as pd
import numpy as np
import datetime as dt
import os
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1. DATA INGESTION
# ==========================================
# Load raw dataset from CSV file
df_raw = pd.read_csv('online_retail_II.csv', encoding='latin1')

# Inspect dataset overview
print("--- Raw Data Info ---")
print(df_raw.info())
print("\n--- First 5 Rows ---")
print(df_raw.head())

# ==========================================
# 2. DATA CLEANING & PREPROCESSING
# ==========================================
# Step 1: Drop records missing Customer ID
df_clean = df_raw.dropna(subset=['Customer ID']).copy()

# Step 2: Cast Customer ID to integer type
df_clean['Customer ID'] = df_clean['Customer ID'].astype(int)

# Step 3: Filter out cancelled transactions (Invoice starting with 'C') and invalid quantities/prices
df_clean = df_clean[~df_clean['Invoice'].astype(str).str.startswith('C')]
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['Price'] > 0)]

# Step 4: Remove exact duplicate entries
df_clean = df_clean.drop_duplicates()

# Step 5: Compute Total Expenditure per line item (TotalPay = Quantity * Price)
df_clean['TotalPay'] = df_clean['Quantity'] * df_clean['Price']

# Step 6: Convert InvoiceDate column to datetime object
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# ==========================================
# 3. RFM FEATURE ENGINEERING
# ==========================================
# Set reference snapshot date (Max invoice date + 1 day)
snapshot_date = df_clean['InvoiceDate'].max() + dt.timedelta(days=1)

# Aggregate metrics at the Customer ID level to calculate Recency, Frequency, and Monetary values
rfm_df = df_clean.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency
    'Invoice': 'nunique',                                     # Frequency
    'TotalPay': 'sum'                                         # Monetary
}).reset_index()

# Rename columns to standard RFM nomenclature
rfm_df.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

print("\n--- Engineered Raw RFM Metrics ---")
print(rfm_df.head())

# ==========================================
# 4. TRANSFORMATION & SCALING
# ==========================================
# Step 1: Apply Log Transformation (log1p) to reduce feature skewness
rfm_log = np.log1p(rfm_df[['Recency', 'Frequency', 'Monetary']])

# Step 2: Standardize features using StandardScaler (mean=0, std=1)
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

# Reconstruct scaled DataFrame for clustering models
rfm_scaled_df = pd.DataFrame(rfm_scaled, columns=['Recency', 'Frequency', 'Monetary'])
rfm_scaled_df['CustomerID'] = rfm_df['CustomerID']

# ==========================================
# 5. EXPORT PROCESSED DATASETS
# ==========================================
# Ensure output directory exists
os.makedirs('dataset', exist_ok=True)

# Export processed datasets for modeling phase
rfm_df.to_csv('dataset/rfm_raw.csv', index=False)
rfm_scaled_df.to_csv('dataset/rfm_scaled.csv', index=False)

print("\nPhase 1 successfully completed. Output files exported to 'dataset/rfm_scaled.csv'.")

--- Raw Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB
None

--- First 5 Rows ---
  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE 